In [1]:
import numpy as np
import pandas as pd
import torch

from resnet_build_imagenet import build_resnet_imagenet


@torch.no_grad()
def exact_size_report(model, seed_bytes_per_layer=8):
    fp_total_bytes = sum(p.numel() * p.element_size() for p in model.parameters())

    binary_total_bytes = 0
    ggd_param_names = set()

    for module_name, m in model.named_modules():
        if m.__class__.__name__ != "Conv2dGGD":
            continue

        g = m.groups
        k = m.kernel_size if isinstance(m.kernel_size, int) else m.kernel_size[0]
        cin_g = m.in_channels // g
        cout_g = m.out_channels // g
        K = cin_g * k * k

        W_flat = m.weight.reshape(g, cout_g, K).float()
        G = m.G.to(dtype=W_flat.dtype, device=W_flat.device)
        W_proj = torch.bmm(G, W_flat.transpose(1, 2))

        W_bits = (W_proj >= 0).transpose(1, 2).reshape(m.out_channels, m.N)
        W_bits = W_bits.to(torch.uint8).cpu().numpy()
        W_packed = np.packbits(W_bits, axis=1)
        Wb_bytes = int(W_packed.nbytes)

        scale_bytes = m.scale.numel() * m.scale.element_size()
        bias_bytes = 0 if m.bias is None else m.bias.numel() * m.bias.element_size()
        seed_bytes = seed_bytes_per_layer

        binary_total_bytes += Wb_bytes + scale_bytes + bias_bytes + seed_bytes

        ggd_param_names.add(f"{module_name}.weight")
        ggd_param_names.add(f"{module_name}.scale")
        if m.bias is not None:
            ggd_param_names.add(f"{module_name}.bias")

    for name, p in model.named_parameters():
        if name in ggd_param_names:
            continue
        binary_total_bytes += p.numel() * p.element_size()

    saved_mb = (fp_total_bytes - binary_total_bytes) / 1e6
    pct_smaller = 100.0 * (1.0 - binary_total_bytes / fp_total_bytes)

    return {
        "full_precision_bytes": fp_total_bytes,
        "binary_export_bytes": binary_total_bytes,
        "full_precision_MB": fp_total_bytes / 1e6,
        "binary_export_MB": binary_total_bytes / 1e6,
        "saved_MB": saved_mb,
        "pct_smaller": pct_smaller,
        "times_smaller": fp_total_bytes / binary_total_bytes,
    }


def readable_size_table_imagenet(
    model_name="18",
    n_scales=range(1, 6),
    seed_bytes_per_layer=8,
):
    rows = []

    for n_scale in n_scales:
        model = build_resnet_imagenet(
            model_name=model_name,
            num_classes=1000,
            in_chans=3,
            use_ggd=True,
            N_scale=float(n_scale),
        )

        stats = exact_size_report(model, seed_bytes_per_layer=seed_bytes_per_layer)

        rows.append({
            "Model": f"ResNet-{model_name} ImageNet",
            "N scale": int(n_scale),
            "Full precision size (MB)": round(stats["full_precision_MB"], 2),
            "Binary size (MB)": round(stats["binary_export_MB"], 2),
            "Space saved (MB)": round(stats["saved_MB"], 2),
            "Smaller by (%)": round(stats["pct_smaller"], 2),
            "Compression factor (x)": round(stats["times_smaller"], 2),
        })

    return pd.DataFrame(rows)


if __name__ == "__main__":
    df = readable_size_table_imagenet(model_name="18", n_scales=range(1, 6))

    print("\nEasy-to-read ImageNet model size table:\n")
    print(df.to_string(index=False))

    print("\nPlain-English summary:\n")
    for _, row in df.iterrows():
        print(
            f"N_scale = {row['N scale']}: "
            f"full precision = {row['Full precision size (MB)']} MB, "
            f"binary = {row['Binary size (MB)']} MB, "
            f"saved = {row['Space saved (MB)']} MB "
            f"({row['Smaller by (%)']}% smaller, "
            f"{row['Compression factor (x)']}x smaller)."
        )


Easy-to-read ImageNet model size table:

             Model  N scale  Full precision size (MB)  Binary size (MB)  Space saved (MB)  Smaller by (%)  Compression factor (x)
ResNet-18 ImageNet        1                     46.78              3.54             43.24           92.43                   13.21
ResNet-18 ImageNet        2                     46.78              4.94             41.84           89.45                    9.48
ResNet-18 ImageNet        3                     46.78              6.33             40.45           86.47                    7.39
ResNet-18 ImageNet        4                     46.78              7.73             39.05           83.48                    6.05
ResNet-18 ImageNet        5                     46.78              9.12             37.66           80.50                    5.13

Plain-English summary:

N_scale = 1: full precision = 46.78 MB, binary = 3.54 MB, saved = 43.24 MB (92.43% smaller, 13.21x smaller).
N_scale = 2: full precision = 46.78 MB, bina